# L1 SCENIC regulatory networks
Input: L1 hypergeom result `/data1st1/junyi/output/sn0827L1/tf_regulon_deg_hypergeom/L1_TF_regulon_deg_hypergeom.csv`
and L1 MAST DEG table `/data2st2/junyi/output/hannan25_L1_mast_degs_fdr_merged_4conditions_7regions.csv`.
One interactive network (html + png) per (comparison x L1 celltype) is written to `figures/regulatory_networks_L1/`.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import glob
import os
import random

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

In [2]:
def build_organic_cluster_layout(graph, tf_list, gene_list, highlight_genes=None):
    import random
    random.seed(42)
    np.random.seed(42)
    
    pos = {}
    
    # 步骤 1：宏观布局 - 将核心 TF 彻底推开（增大半径）
    if len(tf_list) == 1:
        pos[tf_list[0]] = np.array([0.0, 0.0])
    else:
        # 【修改点 1】增大TF之间的距离，从 len(tf_list) * 20 增加到 len(tf_list) * 40
        tf_radius = max(60, len(tf_list) * 20)  
        angles = np.linspace(0, 2 * np.pi, len(tf_list), endpoint=False)
        for i, tf in enumerate(tf_list):
            pos[tf] = np.array([tf_radius * np.cos(angles[i]), tf_radius * np.sin(angles[i])])

    highlight_set = set(highlight_genes) if highlight_genes else set()

    # 步骤 2：微观布局 - 手动制造紧凑但略微拉开的"星爆"簇
    for gene in gene_list:
        parents = list(graph.predecessors(gene))
        if len(parents) == 1:
            # 独有靶基因：紧紧围绕唯一的 TF
            center = pos[parents[0]]
            angle = np.random.uniform(0.5, 2 * np.pi)
            
            # 【修改点 2】调大半径范围（从 0.6~0.7 放大到 0.8~1.2），让TF到Gene的连线更长
            r = np.random.uniform(0.1, 0.2)
            # 高亮基因半径平均 1.5 倍
            if gene in highlight_set:
                r = r * 1.5
            pos[gene] = center + np.array([r * np.cos(angle), r * np.sin(angle)])
        elif len(parents) > 1:
            # 共享靶基因：拉到多个 TF 的几何中心
            center = np.mean([pos[p] for p in parents if p in pos], axis=0)
            noise = np.random.uniform(-0.05, 0.05, 2)  # 增加一些随机性
            if gene in highlight_set:
                noise = noise * 1.5
            pos[gene] = center + noise
        else:
            pos[gene] = np.array([0, 0])

    # 步骤 3：物理微调 (只允许基因动，TF不动)
    pos = nx.spring_layout(
        graph, 
        pos=pos,          
        fixed=tf_list,    # 锁定 TF 节点坐标
        k=0.06,           # 【修改点 3】减小k值，让节点之间更分散
        iterations=30,    # 增加迭代次数
        seed=42
    )
    return pos

def visualize_regulatory_network(metadata_df, importance_col='importance', weight_col='importance_TF2G', figure_size=(15, 9),
                               top_n=100, output_png='figures/network.png', title='SCENIC+ Regulatory Network', df_deg=None,
                               highlight_genes=None, tf_direction_map=None, show_figure=True, save_html=False):
    if metadata_df.empty:
        raise ValueError('metadata_df is empty after filtering.')

    if importance_col not in metadata_df.columns:
        raise KeyError(f'{importance_col} not found in metadata_df')

    if weight_col not in metadata_df.columns:
        metadata_df = metadata_df.copy()
        metadata_df[weight_col] = metadata_df[importance_col]

    df = metadata_df.sort_values(importance_col, ascending=False).head(top_n).copy()

    if df_deg is None:
        df_deg_plot = pd.DataFrame(columns=['gene', 'condition', 'de_coef'])
    else:
        df_deg_plot = df_deg.copy()
        if 'gene' not in df_deg_plot.columns:
            gene_col = next((col for col in ['Gene', 'targetgene', 'TF'] if col in df_deg_plot.columns), None)
            if gene_col is None:
                raise KeyError('df_deg must contain a gene-like column.')
            df_deg_plot = df_deg_plot.rename(columns={gene_col: 'gene'})
        if 'de_coef' not in df_deg_plot.columns:
            coef_col = next((col for col in ['de_coef', 'log2FC'] if col in df_deg_plot.columns), None)
            if coef_col is None:
                raise KeyError('df_deg must contain de_coef or log2FC.')
            df_deg_plot = df_deg_plot.rename(columns={coef_col: 'de_coef'})
        if 'condition' not in df_deg_plot.columns:
            df_deg_plot['condition'] = np.where(df_deg_plot['de_coef'] > 0, 'UP', 'DOWN')
        df_deg_plot['gene'] = df_deg_plot['gene'].astype(str).str.upper()
        df_deg_plot = df_deg_plot.sort_values('de_coef', key=lambda s: s.abs(), ascending=False).drop_duplicates('gene')

    df['TF'] = df['TF'].astype(str).str.upper()
    df['Gene'] = df['Gene'].astype(str).str.upper()

    G = nx.DiGraph()
    for _, row in df.iterrows():
        G.add_edge(
            row['TF'],
            row['Gene'],
            weight=row[weight_col],
            title=f"Weight: {row[weight_col]:.2f}"
        )

    def format_label(name):
        name = str(name).strip()
        return name[:1].upper() + name[1:].lower() if name else name

    degrees = dict(G.degree())
    max_degree = max(degrees.values()) if degrees else 1
    tf_nodes = sorted(df['TF'].unique())
    tf_node_set = set(tf_nodes)
    gene_nodes = sorted([node for node in G.nodes() if node not in tf_node_set])
    deg_lookup = df_deg_plot.set_index('gene')[['condition', 'de_coef']].to_dict('index') if not df_deg_plot.empty else {}

    node_colors = {}
    node_edge_colors = {}
    node_edge_widths = {}

    for node in G.nodes():
        is_tf = node in tf_node_set
        deg_info = deg_lookup.get(node)
        if deg_info is not None and deg_info['condition'] == 'UP':
            fill_color = '#E86A33' if is_tf else '#F3A26D'
            edge_color = '#C9511C'
            edge_width = 1 if is_tf else 0
        elif deg_info is not None and deg_info['condition'] == 'DOWN':
            fill_color = '#3A86C8' if is_tf else '#8EC0E6'
            edge_color = '#2C6BA0'
            edge_width = 0 if is_tf else 0
        else:
            if is_tf:
                # 非DEG的TF：根据 tf_direction_map 上色（浅色），无方向则灰色
                tf_dir = tf_direction_map.get(node) if tf_direction_map else None
                if tf_dir == 'Up':
                    fill_color = '#F3A26D'  # 浅橙色（同UP基因）
                elif tf_dir == 'Down':
                    fill_color = '#8EC0E6'  # 浅蓝色（同DOWN基因）
                else:
                    fill_color = "#B9B9B9"
                edge_color = '#DAA520'
                edge_width = 0
            else:
                fill_color = 'lightgray'
                edge_color = 'lightgray'
                edge_width = 0

        node_colors[node] = fill_color
        node_edge_colors[node] = edge_color
        node_edge_widths[node] = edge_width

    plt.figure(figsize=figure_size)
    pos = build_organic_cluster_layout(G, tf_nodes, gene_nodes, highlight_genes=highlight_genes)    

    node_sizes = []
    highlight_set = set(highlight_genes) if highlight_genes else set()
    for node in G.nodes():
        deg_info = deg_lookup.get(node)
        base_size = 260 if node in tf_node_set else 120
        degree_size = degrees[node] * (900 if node in tf_node_set else 380) / max_degree
        if deg_info is not None:
            fold_change = abs(deg_info['de_coef']) * (520 if node in tf_node_set else 320)
        else:
            fold_change = 0
        total_size = base_size + degree_size + fold_change
        # 高亮基因节点 1.5 倍大小，非高亮基因节点缩小一半
        if node in highlight_set and node not in tf_node_set:
            total_size = total_size * 1.5
        elif node not in tf_node_set:
            total_size = total_size * 0.5
        node_sizes.append(total_size)

    node_shapes = {node: 'h' if node in tf_node_set else 'o' for node in G.nodes()}
    for shape in set(node_shapes.values()):
        nx.draw_networkx_nodes(
            G, pos,
            nodelist=[node for node in G.nodes() if node_shapes[node] == shape],
            node_shape=shape,
            node_color=[node_colors[node] for node in G.nodes() if node_shapes[node] == shape],
            edgecolors=[node_edge_colors[node] for node in G.nodes() if node_shapes[node] == shape],
            linewidths=[node_edge_widths[node] for node in G.nodes() if node_shapes[node] == shape],
            node_size=[node_sizes[index] for index, node in enumerate(G.nodes()) if node_shapes[node] == shape],
            alpha=0.92
        )

    # 绘制边 - 【修改点 4】增加连线弯曲程度
    for source, target, edge_data in G.edges(data=True):
        # 使用弧形连接，rad参数控制弯曲程度（从0.0增加到0.2）
        nx.draw_networkx_edges(
            G, pos, edgelist=[(source, target)],
            width=1.5,
            edge_color="#BEC2C5",
            alpha=0.3,
            arrowstyle='-', 
            connectionstyle='arc3, rad=0.14'  # 增加弯曲程度
        )

    import matplotlib.patheffects as path_effects
    tf_labels = {node: format_label(node) for node in tf_nodes if node in pos}
    # 只有高亮基因才显示文字标签
    gene_labels = {node: format_label(node) for node in gene_nodes if node in pos and node in highlight_set}
    tf_text_labels = nx.draw_networkx_labels(
        G, pos, labels=tf_labels, font_size=10, font_family='sans-serif',
        bbox=dict(facecolor='none', pad=0.1, edgecolor='none')
    )
    gene_text_labels = nx.draw_networkx_labels(
        G, pos, labels=gene_labels, font_size=7, font_family='sans-serif',
        bbox=dict(facecolor='none', pad=0.05, edgecolor='none')
    )
    for _, label in {**tf_text_labels, **gene_text_labels}.items():
        label.set_path_effects([
            path_effects.Stroke(linewidth=2, foreground='white'),
            path_effects.Normal()
        ])

    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    output_dir = os.path.dirname(output_png)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    plt.savefig(output_png, dpi=300)
    if show_figure:
        plt.show()
    else:
        plt.close()

    # optional interactive pyvis html; off by default
    if save_html:
        from pyvis.network import Network
        net = Network(height='800px', width='100%', directed=True)
        net.from_nx(G)
        for node in net.nodes:
            node['label'] = format_label(node['id'])
            if node['id'] in tf_node_set:
                node['shape'] = 'triangle'
                node['size'] = 28
            else:
                node['size'] = 16
        net.barnes_hut(gravity=-25000, central_gravity=0.15, spring_length=140, spring_strength=0.02, damping=0.95)
        net.show_buttons(filter_=['physics'])
        net.save_graph(output_png.replace('.png', '.html'))
        print(f'Interactive visualization saved to {output_png.replace(".png", ".html")}')

# Read inputs


In [3]:
df_hyp = pd.read_csv('/data1st1/junyi/output/sn0827L1/tf_regulon_deg_hypergeom/L1_TF_regulon_deg_hypergeom.csv')
df_hyp['comparison'] = df_hyp['Model']
df_hyp['log2FC'] = df_hyp['AUC_log2FC']   # AUC wilcoxon log2FC
df_hyp['n_up_overlap'] = df_hyp['n_overlap_up']
df_hyp['n_down_overlap'] = df_hyp['n_overlap_down']
df_hyp['up_deg_list'] = df_hyp['overlap_up_genes']
df_hyp['up_deg_list'] = df_hyp['up_deg_list'].fillna('')
df_hyp['down_deg_list'] = df_hyp['overlap_down_genes']
df_hyp['down_deg_list'] = df_hyp['down_deg_list'].fillna('')
df_hyp['Direction'] = np.where(df_hyp['AUC_log2FC'] > 0, 'Up', 'Down')
print(df_hyp.shape, df_hyp.ctname.nunique(), 'celltypes')
df_hyp.head()

(68115, 37) 64 celltypes


,ctname,Region,Celltype,Model,TF,n_target,n_deg_up,n_deg_down,n_overlap_up,n_overlap_down,...,Is_AUC_sig,Is_AUC_sig_up,Is_AUC_sig_down,comparison,log2FC,n_up_overlap,n_down_overlap,up_deg_list,down_deg_list,Direction
0,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Jund,95,629,622,49,2,...,True,True,False,CSRES_M,0.580512,49,2,"ACTB,APLP1,ATP1B1,ATP6V0C,BEX1,BEX2,BMYC,CACNA...","CASKIN1,PSAP",Up
1,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Nfe2l1,64,629,622,33,2,...,True,True,False,CSRES_M,1.182490,33,2,"ATP1A1,ATP1A3,ATP1B1,ATP6V0C,C1QL3,CAMK1D,CHGB...","ARHGEF11,GFAP",Up
2,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Npdc1,171,629,622,57,12,...,True,True,False,CSRES_M,0.294799,57,12,"A830018L16RIK,ACTB,APLP1,ARL6IP5,ATP1B1,ATP5G1...","ATP6V0B,CTSD,FAM173A,GLT8D1,GPRC5B,GRINA,MT3,N...",Up
3,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Nr4a1,104,629,622,31,2,...,True,True,False,CSRES_M,0.537215,31,2,"5730522E02RIK,ANKS1B,APP,CACNA1C,CAR10,CTNNA3,...","ASAP1,SRSF7",Up
4,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Junb,270,629,622,49,0,...,True,True,False,CSRES_M,0.606767,49,0,"5730522E02RIK,A830018L16RIK,APP,ATP1B1,ATP6AP2...",,Up


In [4]:
df_deg_all = pd.read_csv('/data2st2/junyi/output/hannan25_L1_mast_degs_fdr_merged_4conditions_7regions.csv')
df_deg_all['ctname'] = df_deg_all['Region'].astype(str) + '_' + df_deg_all['Class'].astype(str)
df_deg_all['ctname'] = df_deg_all['ctname'].str.replace(' ', '_')
df_deg_all['comparison'] = df_deg_all['Model'].str.replace(' ', '_')
coef_col = 'log2FC'
df_deg_all.head()

,Gene,pval,log2FC,ci.hi,ci.lo,FDR,Bonferroni,celltype.L2,Subclass,Comparison,...,DGL3,Class,mean_UMI_raw_Region_Class,mean_UMI_norm_Region_Class,mean_log1p_Region_Class,mean_UMI_raw_Region_subclass,mean_UMI_norm_Region_subclass,mean_log1p_Region_subclass,ctname,comparison
0,2010300C02Rik,3.976794e-41,0.095962,0.110129,0.081795,5.801530e-41,3.066108e-37,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,CSRES_M
1,4732440D04Rik,4.047720e-63,0.031347,0.038020,0.024675,9.260510e-63,3.120792e-59,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,CSRES_M
2,4933424G06Rik,3.434839e-88,0.065696,0.072870,0.058522,1.131252e-87,2.648261e-84,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,CSRES_M
3,9330175M20Rik,0.000000e+00,0.234866,0.245376,0.224356,0.000000e+00,0.000000e+00,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,CSRES_M
4,A830018L16Rik,1.491574e-80,0.175274,0.193258,0.157290,4.460835e-80,1.150003e-76,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,CSRES_M


In [5]:
# significant regulon-DEG enrichments, plus Is_TF_DEG / AUC annotations from the L1 pipeline
df_tf_target_deg_hypergeom = df_hyp[
    (df_hyp['hypergeom_FDR_up'] < 0.05) | (df_hyp['hypergeom_FDR_down'] < 0.05)
].copy()
print(len(df_tf_target_deg_hypergeom), 'significant (ctname, comparison, TF) rows')
df_tf_target_deg_hypergeom.head()

2615 significant (ctname, comparison, TF) rows


,ctname,Region,Celltype,Model,TF,n_target,n_deg_up,n_deg_down,n_overlap_up,n_overlap_down,...,Is_AUC_sig,Is_AUC_sig_up,Is_AUC_sig_down,comparison,log2FC,n_up_overlap,n_down_overlap,up_deg_list,down_deg_list,Direction
0,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Jund,95,629,622,49,2,...,True,True,False,CSRES_M,0.580512,49,2,"ACTB,APLP1,ATP1B1,ATP6V0C,BEX1,BEX2,BMYC,CACNA...","CASKIN1,PSAP",Up
1,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Nfe2l1,64,629,622,33,2,...,True,True,False,CSRES_M,1.182490,33,2,"ATP1A1,ATP1A3,ATP1B1,ATP6V0C,C1QL3,CAMK1D,CHGB...","ARHGEF11,GFAP",Up
2,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Npdc1,171,629,622,57,12,...,True,True,False,CSRES_M,0.294799,57,12,"A830018L16RIK,ACTB,APLP1,ARL6IP5,ATP1B1,ATP5G1...","ATP6V0B,CTSD,FAM173A,GLT8D1,GPRC5B,GRINA,MT3,N...",Up
3,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Nr4a1,104,629,622,31,2,...,True,True,False,CSRES_M,0.537215,31,2,"5730522E02RIK,ANKS1B,APP,CACNA1C,CAR10,CTNNA3,...","ASAP1,SRSF7",Up
4,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Junb,270,629,622,49,0,...,True,True,False,CSRES_M,0.606767,49,0,"5730522E02RIK,A830018L16RIK,APP,ATP1B1,ATP6AP2...",,Up


In [7]:
# Loop over all comparisons and celltypes, draw one network per (comparison, ctname)
import random

def split_gene_list(value):
    if pd.isna(value) or value == '':
        return []
    return [gene.strip().upper() for gene in str(value).split(',') if gene.strip()]

df_deg_network = df_deg_all.copy()

output_dir = 'figures/regulatory_networks_L1'
os.makedirs(output_dir, exist_ok=True)

comparison_labels = ['CUSUS_M', 'CURES_M', 'CSSUS_M', 'CSRES_M', 'CUSUS_F', 'CURES_F']
all_ctnames = sorted(df_tf_target_deg_hypergeom['ctname'].dropna().unique())
print(f'Total comparisons: {len(comparison_labels)} | total celltypes: {len(all_ctnames)}')

n_done = 0
for comparison_label in comparison_labels:
    for ct_idx, selected_ctname in enumerate(all_ctnames):
        network_source = df_tf_target_deg_hypergeom[
            (df_tf_target_deg_hypergeom['ctname'] == selected_ctname)
            & (df_tf_target_deg_hypergeom['comparison'] == comparison_label)
        ].copy()
        # 刨除NPDC1相关的行
        network_source = network_source[~network_source['TF'].str.contains('Npdc1', case=False, na=False)]
        network_source = pd.concat([
            network_source.sort_values('n_up_overlap', ascending=False).head(5),
            network_source.sort_values('n_down_overlap', ascending=False).head(5)
        ])
        network_source = network_source[
            (network_source['n_up_overlap'] > 0) | (network_source['n_down_overlap'] > 0)
        ].copy()

        if network_source.empty:
            print(f'[skip] no network rows for {selected_ctname} | {comparison_label}')
            continue

        edge_records = []
        for _, row in network_source.iterrows():
            tf_name = str(row['TF']).upper()
            for gene in split_gene_list(row['up_deg_list']):
                edge_records.append({
                    'ctname': row['ctname'],
                    'comparison': row['comparison'],
                    'TF': tf_name,
                    'Gene': gene,
                    'importance': 1,
                    'importance_TF2G': 1,
                    'edge_direction': 'UP'
                })
            for gene in split_gene_list(row['down_deg_list']):
                edge_records.append({
                    'ctname': row['ctname'],
                    'comparison': row['comparison'],
                    'TF': tf_name,
                    'Gene': gene,
                    'importance': 1,
                    'importance_TF2G': -1,
                    'edge_direction': 'DOWN'
                })

        network_edges = pd.DataFrame(edge_records)
        if network_edges.empty:
            print(f'[skip] no edges for {selected_ctname} | {comparison_label}')
            continue
        network_edges = network_edges.drop_duplicates(['ctname', 'comparison', 'TF', 'Gene', 'edge_direction']).copy()

        df_deg_network_ct = df_deg_network[
            (df_deg_network['ctname'] == selected_ctname)
            & (df_deg_network['comparison'] == comparison_label)
        ].copy()
        df_deg_network_ct['gene'] = df_deg_network_ct['Gene'].str.upper()
        df_deg_network_ct['de_coef'] = df_deg_network_ct[coef_col]
        df_deg_network_ct['condition'] = np.where(df_deg_network_ct['de_coef'] > 0, 'UP', 'DOWN')
        df_deg_network_ct = df_deg_network_ct[['gene', 'de_coef', 'condition']].drop_duplicates('gene')

        n_done += 1
        print(f'[{comparison_label} {ct_idx + 1}/{len(all_ctnames)}] {selected_ctname} | edges: {len(network_edges)} | TFs: {network_edges["TF"].nunique()} | genes: {network_edges["Gene"].nunique()}')

        try:
            visualize_regulatory_network(
                network_edges,
                importance_col='importance',
                weight_col='importance_TF2G',
                figure_size=(8, 4),
                top_n=min(300, len(network_edges)),
                output_png=os.path.join(output_dir, f'{selected_ctname}_{comparison_label}_regulatory_network.png'),
                title='',
                highlight_genes=random.sample(network_edges['Gene'].unique().tolist(), min(15, network_edges['Gene'].nunique())),
                df_deg=df_deg_network_ct,
                tf_direction_map=dict(zip(network_source['TF'], network_source['Direction'])),
                show_figure=False
            )
        except Exception as e:
            print(f'[error] {selected_ctname} | {comparison_label}: {e}')
            plt.close('all')
            continue

print(f'All comparison x celltype networks done, {n_done} drawn.')

Total comparisons: 6 | total celltypes: 58
[CUSUS_M 1/58] AMY_Astrocyte | edges: 117 | TFs: 5 | genes: 109
[CUSUS_M 2/58] AMY_GABA | edges: 243 | TFs: 6 | genes: 156
[CUSUS_M 3/58] AMY_Glut | edges: 8 | TFs: 1 | genes: 8
[skip] no network rows for AMY_Microglia | CUSUS_M
[CUSUS_M 5/58] AMY_OPC | edges: 15 | TFs: 2 | genes: 15
[CUSUS_M 6/58] AMY_Oligo | edges: 497 | TFs: 8 | genes: 293
[skip] no network rows for AMY_Vascular | CUSUS_M
[CUSUS_M 8/58] HPF_Astrocyte | edges: 170 | TFs: 4 | genes: 159
[CUSUS_M 9/58] HPF_Epen | edges: 9 | TFs: 2 | genes: 7
[CUSUS_M 10/58] HPF_GABA | edges: 107 | TFs: 7 | genes: 88
[CUSUS_M 11/58] HPF_Glut | edges: 44 | TFs: 3 | genes: 34
[CUSUS_M 12/58] HPF_Microglia | edges: 10 | TFs: 2 | genes: 9
[CUSUS_M 13/58] HPF_OPC | edges: 14 | TFs: 2 | genes: 12
[CUSUS_M 14/58] HPF_Oligo | edges: 307 | TFs: 6 | genes: 238
[skip] no network rows for HPF_Vascular | CUSUS_M
[CUSUS_M 16/58] HY_Astrocyte | edges: 245 | TFs: 7 | genes: 191
[skip] no network rows for HY_Ch

In [ ]:
# Cell-type-specific regulon dot plot
#   1) for every (TF, comparison), assign the cell type whose DEGs the TF's regulon
#      targets are most enriched in (max -log10 hypergeom FDR up/down)
#   2) keep only regulons that are differential (AUC wilcoxon FDR<0.05) in at least one L1 class
#   3) order rows by target cell type -> mean |log2FC|, draw coloured group blocks + side labels
#   4) each cell shows the AUC log2FC (RdBu_r) and pct_nz_group (dot size), only if FDR<0.05
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

COMPARISON = 'SUS_M'                       # one of: SUS_M, SUS_F, RES_M, RES_F, CSDS_M, CSRES_M
CT_ORDER = ['Astrocyte', 'Glut', 'GABA', 'Microglia', 'OPC', 'Oligo']
MIN_OVERLAP = 3                            # require this many overlap genes for a TF to count as cell-type-specific
TOP_N_PER_BLOCK = 10                       # show at most this many regulons per cell-type block

# ----- 1) AUC wilcoxon results, restricted to the chosen comparison and 6 L1 classes -----
auc = (pd.read_csv("/data2/junyi/stg0901/scenic_wilcoxon_L1/combined_wilcoxon_DEGs.csv")
       .query("comparison == @COMPARISON and `celltype.L1` in @CT_ORDER"))
auc['TF'] = auc['TF'].str.replace("(+)", "", regex=False)
auc['regulon'] = auc['TF'] + "(+)"

# ----- 2) cell-type specificity: per TF, which class's DEGs does the regulon target best? -----
hyp = pd.read_csv('/data1st1/junyi/output/sn0827L1/tf_regulon_deg_hypergeom/L1_TF_regulon_deg_hypergeom.csv')
# AUC table names: SUS_M / RES_M / CSDS_M / CSRES_M
# hypergeom table names: CUSUS_M / CURES_M / CSSUS_M / CSRES_M
# build a mapping so the same logical comparison joins across both
comparison_to_model = {
    'SUS_M': 'CUSUS_M', 'SUS_F': 'CUSUS_F',
    'RES_M': 'CURES_M', 'RES_F': 'CURES_F',
    'CSDS_M': 'CSSUS_M', 'CSRES_M': 'CSRES_M',
}
hyp_combo = hyp[hyp['Model'] == comparison_to_model[COMPARISON]].copy()
hyp_combo['score'] = np.where(
    hyp_combo['hypergeom_FDR_up'] < 0.05,
    -np.log10(hyp_combo['hypergeom_FDR_up'].replace(0, 1e-300)),
    np.where(
        hyp_combo['hypergeom_FDR_down'] < 0.05,
        -np.log10(hyp_combo['hypergeom_FDR_down'].replace(0, 1e-300)),
        0.0,
    ),
)
hyp_combo = hyp_combo[hyp_combo['score'] > 0]
# require minimum overlap to avoid spurious specificity from a single gene
hyp_combo = hyp_combo[
    (hyp_combo['n_overlap_up'] >= MIN_OVERLAP) | (hyp_combo['n_overlap_down'] >= MIN_OVERLAP)
]
best = (hyp_combo.sort_values('score', ascending=False)
        .groupby('TF', as_index=False).head(1)[['TF', 'Celltype']])
best = best.rename(columns={'Celltype': 'target_ct'})

# ----- 3) regulons must be differential somewhere to appear at all -----
sig_set = set(auc[auc.FDR < 0.05]['regulon'].str.replace("(+)", "", regex=False))
auc_plot = auc[auc['TF'].isin(sig_set) & auc['celltype.L1'].isin(CT_ORDER)].copy()
auc_plot = auc_plot.merge(best, left_on='TF', right_on='TF', how='left')
auc_plot = auc_plot[auc_plot['target_ct'].isin(CT_ORDER)]

# ranking: for each cell-type block, top N regulons by mean |log2FC| in that target class
def _top_in_block(df, n):
    return (df.assign(abs_l2=df['log2FC'].abs())
              .query("`celltype.L1` == target_ct")
              .groupby('TF')['abs_l2'].mean()
              .sort_values(ascending=False).head(n).index.tolist())

selected = []
for ct in CT_ORDER:
    selected += _top_in_block(auc_plot[auc_plot['target_ct'] == ct], TOP_N_PER_BLOCK)
if not selected:
    raise ValueError(f"No regulons pass the filters for {COMPARISON}")

auc_plot = auc_plot[auc_plot['TF'].isin(selected)]
# final order: by block, then by within-block |log2FC|
auc_plot['__o'] = auc_plot['target_ct'].map({c: i for i, c in enumerate(CT_ORDER)})
order = (auc_plot.assign(__abs=auc_plot['log2FC'].abs())
         .sort_values(['__o', '__abs'], ascending=[True, False])['TF'].drop_duplicates().tolist())

# ----- 4) build the three matrices and plot -----
mat_l2 = (auc_plot.pivot_table(index='TF', columns='celltype.L1',
                               values='log2FC', aggfunc='mean')
          .reindex(index=order, columns=CT_ORDER))
mat_pct = (auc_plot.pivot_table(index='TF', columns='celltype.L1',
                                values='pct_nz_group', aggfunc='mean')
           .reindex(index=order, columns=CT_ORDER))
mat_fdr = (auc_plot.pivot_table(index='TF', columns='celltype.L1',
                                values='FDR', aggfunc='min')
           .reindex(index=order, columns=CT_ORDER))
mask = mat_fdr < 0.05

# group index for each row -> block background colour and side label
ct_colours = {
    'Astrocyte': '#7A4FB1', 'Glut': '#33A02C', 'GABA': '#1F78B4',
    'Microglia': '#E31A1C', 'OPC': '#FDBF6F', 'Oligo': '#7B9FBE',
}
row_groups = auc_plot.drop_duplicates('TF').set_index('TF').loc[order, 'target_ct']
row_colour = row_groups.map(ct_colours).values

n_rows, n_cols = mat_l2.shape
fig, ax = plt.subplots(figsize=(n_cols * 1.5, max(3, n_rows * 0.32)))

vmax = float(np.nanmax(np.abs(mat_l2.values))) if not mat_l2.isna().all().all() else 1.0
vmax = max(vmax, 0.1)
im = ax.imshow(mat_l2.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

# scatter dots (only in FDR<0.05 cells)
rng = np.random.default_rng(0)
for i in range(n_rows):
    for j in range(n_cols):
        if not mask.iloc[i, j]:
            continue
        pct = mat_pct.iloc[i, j]
        if pd.isna(pct):
            continue
        ax.scatter(j + rng.uniform(-0.06, 0.06), i + rng.uniform(-0.06, 0.06),
                   s=20 + pct * 220, c='black', alpha=0.85,
                   edgecolors='none', zorder=3)

# block background: subtle horizontal band per cell-type group
block_starts = [0]
for k in range(1, n_rows):
    block_starts.append(block_starts[-1] + (row_groups.iloc[k] != row_groups.iloc[k-1]))
unique_blocks = []
for i, ct in enumerate(row_groups):
    if i == 0 or row_groups.iloc[i] != row_groups.iloc[i-1]:
        unique_blocks.append((i, ct))
unique_blocks.append((n_rows, None))
for (s, ct), (e, _) in zip(unique_blocks[:-1], unique_blocks[1:]):
    if e - s > 1:                           # only draw separator if block has >1 row
        ax.axhline(e - 0.5, color='white', lw=2, zorder=4)

# right-side group label (rotated, coloured like the column header)
for i, ct in row_groups.items():
    pass
last_in_block = {}
for i, ct in enumerate(row_groups):
    last_in_block[ct] = i  # overwrite; ends up holding the last occurrence
for ct, idx in last_in_block.items():
    ax.text(n_cols - 0.3, idx, ct,
            color=ct_colours.get(ct, 'black'), fontsize=8, fontweight='bold',
            ha='left', va='center', clip_on=False)

# column header colour bar
for j, ct in enumerate(CT_ORDER):
    ax.add_patch(mpatches.Rectangle((j - 0.5, -1.4), 1, 0.9,
                                    color=ct_colours.get(ct, 'lightgray'),
                                    clip_on=False, zorder=2))
    ax.text(j, -0.95, ct, ha='center', va='center', fontsize=8, color='white',
            fontweight='bold', clip_on=False)

ax.set_yticks(range(n_rows))
ax.set_yticklabels([f'{r}(+)' for r in order], fontsize=7)
ax.set_xticks(range(n_cols))
ax.set_xticklabels([''] * n_cols)
ax.tick_params(axis='x', which='both', length=0)
ax.set_ylim(n_rows - 0.5, -1.4)
ax.xaxis.set_visible(False)

# subtle block background colour tint on the y-axis (highlight the target cell type column)
for i, col in enumerate(row_colour):
    ax.add_patch(mpatches.Rectangle((-0.5, i - 0.5), 0.15, 1.0,
                                    color=col, alpha=0.55, clip_on=False, zorder=2))

cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
cbar.set_label('AUC log2FC (wilcoxon)', fontsize=8)
cbar.ax.tick_params(labelsize=7)

size_legend = [Line2D([0], [0], marker='o', color='w',
                       markerfacecolor='black', markersize=4 + s, alpha=0.85,
                       label=f'{int(s*100)}%')
               for s in (0.10, 0.30, 0.50)]
ax.legend(handles=size_legend, title='pct_nz_group', loc='lower right',
          bbox_to_anchor=(1.0, -0.18), ncol=3, frameon=False, fontsize=7, title_fontsize=7)

ax.set_title(f'Cell-type-specific regulons across L1 cell types  |  {COMPARISON} vs CON',
             fontsize=10, pad=30)
plt.tight_layout()
out = f"figures/regulatory_networks_L1/dotplot_{COMPARISON}_celltype_specific.png"
os.makedirs(os.path.dirname(out), exist_ok=True)
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print("saved:", out)